In [38]:
import pandas as pd
from pathlib import Path
from collections import Counter

In [40]:
DATA_DIR = Path("../training_datasets")

S1_PATH = DATA_DIR / "train_source1.tsv"
S2_PATH = DATA_DIR / "train_source2.tsv"
S3_PATH = DATA_DIR / "train_source3.tsv"
GT_PATH = DATA_DIR / "train_ground_truth.tsv"

CHUNK_SIZE = 100_000

In [41]:
def profile_source(path, chunk_size=100_000):
    total_rows = 0

    missing_counts = Counter()
    country_counts = Counter()

    unique_ids = set()
    duplicate_ids = 0

    name_lengths = []
    address_lengths = []

    unique_names = set()
    unique_addresses = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=chunk_size
    ):
        total_rows += len(chunk)

        # Missing values
        missing_counts.update(
            chunk.isna().sum().to_dict()
        )

        # Countries
        country_counts.update(
            chunk["country"].fillna("<MISSING>").value_counts().to_dict()
        )

        # Entity IDs
        ids = chunk["entity_id"].dropna()

        duplicate_ids += ids.duplicated().sum()

        # Check duplicates across chunks too
        for entity_id in ids:
            if entity_id in unique_ids:
                duplicate_ids += 1
            else:
                unique_ids.add(entity_id)

        # Name statistics
        names = chunk["business_name"].fillna("")

        name_lengths.extend(names.str.len().tolist())
        unique_names.update(names[names != ""])

        # Address statistics
        addresses = chunk["business_address"].fillna("")

        address_lengths.extend(addresses.str.len().tolist())
        unique_addresses.update(addresses[addresses != ""])

    print(f"Rows: {total_rows:,}")
    print(f"Columns: 4")

    print("\nMissing values:")
    for column, count in missing_counts.items():
        print(f"  {column}: {count:,}")

    print("\nEntity IDs:")
    print(f"  Unique IDs: {len(unique_ids):,}")
    print(f"  Duplicate IDs: {duplicate_ids:,}")

    print("\nCountries:")
    for country, count in country_counts.most_common():
        print(f"  {country}: {count:,}")

    print("\nBusiness name:")
    print(f"  Unique names: {len(unique_names):,}")
    print(f"  Average length: {sum(name_lengths) / len(name_lengths):.2f}")
    print(f"  Min length: {min(name_lengths)}")
    print(f"  Max length: {max(name_lengths)}")

    print("\nBusiness address:")
    print(f"  Unique addresses: {len(unique_addresses):,}")
    print(f"  Average length: {sum(address_lengths) / len(address_lengths):.2f}")
    print(f"  Min length: {min(address_lengths)}")
    print(f"  Max length: {max(address_lengths)}")

    return {
        "rows": total_rows,
        "unique_ids": len(unique_ids),
        "duplicate_ids": duplicate_ids,
        "country_counts": country_counts,
        "missing_counts": missing_counts,
    }

In [42]:
s1_profile = profile_source(S1_PATH)

Rows: 2,206,821
Columns: 4

Missing values:
  entity_id: 0
  business_name: 0
  business_address: 0
  country: 0

Entity IDs:
  Unique IDs: 2,206,821
  Duplicate IDs: 0

Countries:
  US: 1,323,633
  India: 883,188

Business name:
  Unique names: 1,539,229
  Average length: 24.03
  Min length: 3
  Max length: 105

Business address:
  Unique addresses: 2,130,606
  Average length: 52.07
  Min length: 11
  Max length: 256


In [43]:
s2_profile = profile_source(S2_PATH)

Rows: 5,034,616
Columns: 4

Missing values:
  entity_id: 0
  business_name: 2
  business_address: 168,967
  country: 0

Entity IDs:
  Unique IDs: 5,034,616
  Duplicate IDs: 0

Countries:
  US: 3,016,817
  India: 2,017,799

Business name:
  Unique names: 4,402,008
  Average length: 25.10
  Min length: 0
  Max length: 104

Business address:
  Unique addresses: 4,337,261
  Average length: 46.23
  Min length: 0
  Max length: 249


In [44]:
s3_profile = profile_source(S3_PATH)

Rows: 5,285,603
Columns: 4

Missing values:
  entity_id: 0
  business_name: 13
  business_address: 175,916
  country: 0

Entity IDs:
  Unique IDs: 5,285,603
  Duplicate IDs: 0

Countries:
  US: 3,170,056
  India: 2,115,547

Business name:
  Unique names: 4,651,608
  Average length: 25.20
  Min length: 0
  Max length: 123

Business address:
  Unique addresses: 4,632,764
  Average length: 46.71
  Min length: 0
  Max length: 240


# Profiling Ground Truth Function

In [45]:
def get_entity_ids(path, chunk_size=100_000):
    ids = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=["entity_id"],
        chunksize=chunk_size
    ):
        ids.update(chunk["entity_id"].dropna())

    return ids

In [46]:
s2_ids = get_entity_ids(S2_PATH)
s3_ids = get_entity_ids(S3_PATH)

print(f"S2 IDs: {len(s2_ids):,}")
print(f"S3 IDs: {len(s3_ids):,}")

S2 IDs: 5,034,616
S3 IDs: 5,285,603


In [47]:
def profile_ground_truth(path, s2_ids, s3_ids):
    total_rows = 0

    singleton_count = 0
    match_count_distribution = Counter()

    s2_match_count = 0
    s3_match_count = 0
    unknown_match_count = 0

    source1_ids = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=100_000
    ):
        total_rows += len(chunk)

        # Source 1 IDs
        source1_ids.update(
            chunk["source1_entity_id"].dropna()
        )

        for matched_ids in chunk["matched_entity_ids"].fillna(""):
            matched_ids = matched_ids.strip()

            # Singleton
            if not matched_ids:
                singleton_count += 1
                match_count_distribution[0] += 1
                continue

            ids = [
                entity_id.strip()
                for entity_id in matched_ids.split(",")
                if entity_id.strip()
            ]

            match_count_distribution[len(ids)] += 1

            for entity_id in ids:
                if entity_id in s2_ids:
                    s2_match_count += 1

                elif entity_id in s3_ids:
                    s3_match_count += 1

                else:
                    unknown_match_count += 1

    print(f"Rows: {total_rows:,}")

    print("\nUnique Source 1 IDs:")
    print(f"  {len(source1_ids):,}")

    print("\nMatch count distribution:")

    for count, frequency in sorted(match_count_distribution.items()):
        print(f"  {count} matches: {frequency:,}")

    print("\nSingletons:")
    print(f"  {singleton_count:,}")
    print(f"  Singleton rate: {singleton_count / total_rows:.2%}")

    print("\nMatches by source:")
    print(f"  S2: {s2_match_count:,}")
    print(f"  S3: {s3_match_count:,}")
    print(f"  Unknown IDs: {unknown_match_count:,}")

    return {
        "rows": total_rows,
        "unique_source1_ids": len(source1_ids),
        "singleton_count": singleton_count,
        "singleton_rate": singleton_count / total_rows,
        "match_distribution": match_count_distribution,
        "s2_matches": s2_match_count,
        "s3_matches": s3_match_count,
        "unknown_matches": unknown_match_count,
    }

In [48]:
gt_profile = profile_ground_truth(
    GT_PATH,
    s2_ids,
    s3_ids
)

Rows: 2,206,821

Unique Source 1 IDs:
  2,206,821

Match count distribution:
  0 matches: 123,247
  1 matches: 119,157
  2 matches: 375,212
  3 matches: 530,841
  4 matches: 484,115
  5 matches: 321,957
  6 matches: 164,868
  7 matches: 63,968
  8 matches: 18,680
  9 matches: 4,205
  10 matches: 534
  11 matches: 37

Singletons:
  123,247
  Singleton rate: 5.58%

Matches by source:
  S2: 3,693,619
  S3: 3,944,746
  Unknown IDs: 0


# Classifying Ground Truth Matches by Source

In [49]:
from collections import Counter

def profile_match_sources(path, s2_ids, s3_ids, chunk_size=100_000):
    categories = Counter()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=chunk_size
    ):
        for matched_ids in chunk["matched_entity_ids"].fillna(""):

            matched_ids = matched_ids.strip()

            # No matches
            if not matched_ids:
                categories["neither"] += 1
                continue

            ids = [
                entity_id.strip()
                for entity_id in matched_ids.split(",")
                if entity_id.strip()
            ]

            has_s2 = False
            has_s3 = False

            for entity_id in ids:
                if entity_id in s2_ids:
                    has_s2 = True
                elif entity_id in s3_ids:
                    has_s3 = True

            if has_s2 and has_s3:
                categories["both"] += 1
            elif has_s2:
                categories["s2_only"] += 1
            elif has_s3:
                categories["s3_only"] += 1
            else:
                categories["unknown"] += 1

    total = sum(categories.values())

    print(f"Total S1 businesses: {total:,}")

    print("\nMatch source categories:")

    for category in ["s2_only", "s3_only", "both", "neither", "unknown"]:
        count = categories[category]
        percentage = count / total * 100

        print(
            f"  {category:10} → "
            f"{count:>10,} ({percentage:.2f}%)"
        )

    return categories

In [50]:
source_categories = profile_match_sources(
    GT_PATH,
    s2_ids,
    s3_ids
)

Total S1 businesses: 2,206,821

Match source categories:
  s2_only    →    143,029 (6.48%)
  s3_only    →    164,498 (7.45%)
  both       →  1,776,047 (80.48%)
  neither    →    123,247 (5.58%)
  unknown    →          0 (0.00%)


# Disjointness of Source 2 and Source 3

In [52]:
def get_ids(path):
    ids = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id"],
        chunksize=CHUNK_SIZE,
        dtype={"entity_id": "string"}
    ):
        ids.update(chunk["entity_id"].dropna())

    return ids


print("Reading S2 IDs...")
s2_ids = get_ids(S2_PATH)

print("Reading S3 IDs...")
s3_ids = get_ids(S3_PATH)

print("\n--- DISJOINTNESS CHECK ---")

print(f"S2 unique IDs: {len(s2_ids):,}")
print(f"S3 unique IDs: {len(s3_ids):,}")

overlap = s2_ids & s3_ids

print(f"S2 ∩ S3: {len(overlap):,}")

print(f"S2-only: {len(s2_ids - s3_ids):,}")
print(f"S3-only: {len(s3_ids - s2_ids):,}")

print(
    f"\nOverlap rate relative to S2: "
    f"{len(overlap) / len(s2_ids) * 100:.2f}%"
)

print(
    f"Overlap rate relative to S3: "
    f"{len(overlap) / len(s3_ids) * 100:.2f}%"
)

Reading S2 IDs...
Reading S3 IDs...

--- DISJOINTNESS CHECK ---
S2 unique IDs: 5,034,616
S3 unique IDs: 5,285,603
S2 ∩ S3: 0
S2-only: 5,034,616
S3-only: 5,285,603

Overlap rate relative to S2: 0.00%
Overlap rate relative to S3: 0.00%


In [53]:
# ============================================================
# COUNTRY CONSISTENCY CHECK
# S1 ↔ S2 and S1 ↔ S3 true matches from ground truth
# ============================================================

import pandas as pd

print("Loading ground truth...")

# Only load the columns we need
gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    usecols=["source1_entity_id", "matched_entity_ids"],
    dtype=str
)

print(f"Ground truth rows: {len(gt):,}")


# ------------------------------------------------------------
# 1. EXPLODE GROUND TRUTH INTO INDIVIDUAL MATCH PAIRS
# ------------------------------------------------------------

pairs = []

for _, row in gt.iterrows():

    s1_id = row["source1_entity_id"]
    matched = row["matched_entity_ids"]

    if pd.isna(s1_id) or pd.isna(matched):
        continue

    for matched_id in str(matched).split(","):

        matched_id = matched_id.strip()

        if matched_id.startswith("S2-"):
            pairs.append((s1_id, matched_id, "S2"))

        elif matched_id.startswith("S3-"):
            pairs.append((s1_id, matched_id, "S3"))


pairs = pd.DataFrame(
    pairs,
    columns=["s1_id", "matched_id", "source"]
)

print(f"Total true match pairs: {len(pairs):,}")
print(f"S1 ↔ S2 pairs: {(pairs['source'] == 'S2').sum():,}")
print(f"S1 ↔ S3 pairs: {(pairs['source'] == 'S3').sum():,}")


# ------------------------------------------------------------
# 2. GET ONLY THE IDS WE NEED
# ------------------------------------------------------------

s1_ids = set(pairs["s1_id"])
s2_ids = set(pairs.loc[pairs["source"] == "S2", "matched_id"])
s3_ids = set(pairs.loc[pairs["source"] == "S3", "matched_id"])


# ------------------------------------------------------------
# 3. LOOK UP COUNTRIES FROM EACH SOURCE
#    Chunked so we don't unnecessarily load 5M+ rows
# ------------------------------------------------------------

def load_country_lookup(path, wanted_ids, label):

    lookup = {}

    print(f"\nReading {label} countries...")

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country"],
        dtype=str,
        chunksize=CHUNK_SIZE
    ):

        mask = chunk["entity_id"].isin(wanted_ids)

        if mask.any():

            selected = chunk.loc[mask, ["entity_id", "country"]]

            for entity_id, country in selected.itertuples(index=False):
                lookup[entity_id] = country

    print(f"{label} IDs found: {len(lookup):,} / {len(wanted_ids):,}")

    return lookup


s1_country = load_country_lookup(
    S1_PATH,
    s1_ids,
    "S1"
)

s2_country = load_country_lookup(
    S2_PATH,
    s2_ids,
    "S2"
)

s3_country = load_country_lookup(
    S3_PATH,
    s3_ids,
    "S3"
)


# ------------------------------------------------------------
# 4. ATTACH COUNTRIES TO MATCH PAIRS
# ------------------------------------------------------------

pairs["s1_country"] = pairs["s1_id"].map(s1_country)

pairs["matched_country"] = pairs.apply(
    lambda row:
        s2_country.get(row["matched_id"])
        if row["source"] == "S2"
        else s3_country.get(row["matched_id"]),
    axis=1
)


# ------------------------------------------------------------
# 5. NORMALIZE COUNTRY VALUES FOR COMPARISON
# ------------------------------------------------------------

pairs["s1_country_norm"] = (
    pairs["s1_country"]
    .fillna("")
    .str.strip()
    .str.lower()
)

pairs["matched_country_norm"] = (
    pairs["matched_country"]
    .fillna("")
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# 6. CHECK CONSISTENCY
# ------------------------------------------------------------

# Only compare rows where BOTH countries are available
pairs["countries_available"] = (
    (pairs["s1_country_norm"] != "") &
    (pairs["matched_country_norm"] != "")
)

pairs["country_match"] = (
    pairs["countries_available"] &
    (
        pairs["s1_country_norm"]
        == pairs["matched_country_norm"]
    )
)


# ------------------------------------------------------------
# 7. PRINT RESULTS
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("COUNTRY CONSISTENCY")
print("=" * 60)


for source in ["S2", "S3"]:

    subset = pairs[pairs["source"] == source]

    available = subset["countries_available"].sum()

    mismatches = (
        subset.loc[subset["countries_available"], "country_match"]
        == False
    ).sum()

    matches = (
        subset.loc[subset["countries_available"], "country_match"]
        == True
    ).sum()

    missing = len(subset) - available

    mismatch_rate = (
        mismatches / available * 100
        if available > 0
        else 0
    )

    print(f"\nS1 ↔ {source}")
    print("-" * 30)
    print(f"True match pairs:       {len(subset):,}")
    print(f"Countries available:    {available:,}")
    print(f"Country matches:        {matches:,}")
    print(f"Country mismatches:     {mismatches:,}")
    print(f"Missing country values: {missing:,}")
    print(f"Mismatch rate:          {mismatch_rate:.4f}%")


# ------------------------------------------------------------
# 8. SHOW ACTUAL MISMATCHES
# ------------------------------------------------------------

mismatches = pairs[
    pairs["countries_available"] &
    (~pairs["country_match"])
]

print("\n")
print("=" * 60)
print("COUNTRY MISMATCH DETAILS")
print("=" * 60)

if len(mismatches) == 0:

    print("No country mismatches found.")

else:

    print(mismatches[
        [
            "s1_id",
            "s1_country",
            "matched_id",
            "matched_country",
            "source"
        ]
    ].to_string(index=False))

Loading ground truth...
Ground truth rows: 2,206,821
Total true match pairs: 7,638,365
S1 ↔ S2 pairs: 3,693,619
S1 ↔ S3 pairs: 3,944,746

Reading S1 countries...
S1 IDs found: 2,083,574 / 2,083,574

Reading S2 countries...
S2 IDs found: 3,693,619 / 3,693,619

Reading S3 countries...
S3 IDs found: 3,944,746 / 3,944,746


COUNTRY CONSISTENCY

S1 ↔ S2
------------------------------
True match pairs:       3,693,619
Countries available:    3,693,619
Country matches:        3,693,619
Country mismatches:     0
Missing country values: 0
Mismatch rate:          0.0000%

S1 ↔ S3
------------------------------
True match pairs:       3,944,746
Countries available:    3,944,746
Country matches:        3,944,746
Country mismatches:     0
Missing country values: 0
Mismatch rate:          0.0000%


COUNTRY MISMATCH DETAILS
No country mismatches found.


# Name Noice Check

In [55]:
# ============================================================
# 3. NAME NOISE ANALYSIS
# ============================================================

import re

# ------------------------------------------------------------
# 1. LOAD GROUND TRUTH
# ------------------------------------------------------------

print("Loading ground truth...")

gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    usecols=["source1_entity_id", "matched_entity_ids"],
    dtype=str
)

# ------------------------------------------------------------
# 2. EXPLODE MATCHES
# ------------------------------------------------------------

pairs = []

for row in gt.itertuples(index=False):

    s1_id = str(row.source1_entity_id)
    matched_ids = str(row.matched_entity_ids)

    for matched_id in matched_ids.split(","):

        matched_id = matched_id.strip()

        if matched_id.startswith("S2-"):
            pairs.append(
                (s1_id, matched_id[3:], "S2")
            )

        elif matched_id.startswith("S3-"):
            pairs.append(
                (s1_id, matched_id[3:], "S3")
            )

pairs = pd.DataFrame(
    pairs,
    columns=["s1_id", "matched_id", "source"]
)

# Force IDs to strings
pairs["s1_id"] = pairs["s1_id"].astype(str)
pairs["matched_id"] = pairs["matched_id"].astype(str)

print(f"Total true match pairs: {len(pairs):,}")


# ------------------------------------------------------------
# 3. LOAD S1 NAMES
# ------------------------------------------------------------

def load_names(path, wanted_ids):

    names = {}

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_name"],
        dtype={"entity_id": str, "business_name": str},
        chunksize=CHUNK_SIZE
    ):

        chunk["entity_id"] = chunk["entity_id"].str.strip()

        selected = chunk[
            chunk["entity_id"].isin(wanted_ids)
        ]

        for entity_id, name in selected.itertuples(index=False):
            names[entity_id] = name

    return names


s1_ids = set(pairs["s1_id"])

print("\nLoading S1 business names...")

s1_names = load_names(
    S1_PATH,
    s1_ids
)

print(f"S1 names loaded: {len(s1_names):,}")


# ------------------------------------------------------------
# 4. LOAD S2 NAMES
# ------------------------------------------------------------

s2_ids = set(
    pairs.loc[
        pairs["source"] == "S2",
        "matched_id"
    ]
)

print("\nLoading S2 business names...")

s2_names = load_names(
    S2_PATH,
    s2_ids
)

print(f"S2 names loaded: {len(s2_names):,}")


# ------------------------------------------------------------
# 5. LOAD S3 NAMES
# ------------------------------------------------------------

s3_ids = set(
    pairs.loc[
        pairs["source"] == "S3",
        "matched_id"
    ]
)

print("\nLoading S3 business names...")

s3_names = load_names(
    S3_PATH,
    s3_ids
)

print(f"S3 names loaded: {len(s3_names):,}")


# ------------------------------------------------------------
# 6. ATTACH NAMES
# ------------------------------------------------------------

pairs["s1_name"] = pairs["s1_id"].map(s1_names)

pairs["matched_name"] = pairs["matched_id"].map(
    lambda x: s2_names.get(x) or s3_names.get(x)
)


# ------------------------------------------------------------
# 7. CHECK WHAT WE ACTUALLY LOADED
# ------------------------------------------------------------

print("\nName availability:")

print(
    "S1 names:",
    pairs["s1_name"].notna().sum()
)

print(
    "Matched names:",
    pairs["matched_name"].notna().sum()
)


# ------------------------------------------------------------
# 8. REMOVE MISSING / EMPTY NAMES
# ------------------------------------------------------------

pairs = pairs.dropna(
    subset=["s1_name", "matched_name"]
).copy()

pairs = pairs[
    (pairs["s1_name"].str.strip() != "") &
    (pairs["matched_name"].str.strip() != "")
].copy()

print(
    f"\nPairs with both names available: "
    f"{len(pairs):,}"
)


# ------------------------------------------------------------
# 9. NORMALIZATION
# ------------------------------------------------------------

def lowercase_name(name):
    return str(name).lower().strip()


def basic_normalize(name):

    name = str(name).lower()

    # Replace punctuation with spaces
    name = re.sub(r"[^\w\s]", " ", name)

    # Collapse whitespace
    name = re.sub(r"\s+", " ", name)

    return name.strip()


COMMON_SUFFIXES = {
    "inc",
    "incorporated",
    "ltd",
    "limited",
    "llc",
    "corp",
    "corporation",
    "co",
    "company",
    "pvt",
    "private"
}


def normalize_without_suffix(name):

    words = basic_normalize(name).split()

    while words and words[-1] in COMMON_SUFFIXES:
        words.pop()

    return " ".join(words)


pairs["s1_lower"] = pairs["s1_name"].apply(
    lowercase_name
)

pairs["matched_lower"] = pairs["matched_name"].apply(
    lowercase_name
)

pairs["s1_basic"] = pairs["s1_name"].apply(
    basic_normalize
)

pairs["matched_basic"] = pairs["matched_name"].apply(
    basic_normalize
)

pairs["s1_no_suffix"] = pairs["s1_name"].apply(
    normalize_without_suffix
)

pairs["matched_no_suffix"] = pairs["matched_name"].apply(
    normalize_without_suffix
)


# ------------------------------------------------------------
# 10. CALCULATE MATCH RATES
# ------------------------------------------------------------

total = len(pairs)

exact = (
    pairs["s1_name"] == pairs["matched_name"]
).sum()

lower = (
    pairs["s1_lower"] == pairs["matched_lower"]
).sum()

basic = (
    pairs["s1_basic"] == pairs["matched_basic"]
).sum()

suffix = (
    pairs["s1_no_suffix"] == pairs["matched_no_suffix"]
).sum()


print("\n")
print("=" * 60)
print("NAME NOISE ANALYSIS")
print("=" * 60)

print(f"\nPairs analyzed: {total:,}")

if total > 0:

    print(
        f"\nExact matches:              "
        f"{exact:,} ({exact / total * 100:.2f}%)"
    )

    print(
        f"Lowercase matches:         "
        f"{lower:,} ({lower / total * 100:.2f}%)"
    )

    print(
        f"Basic normalized matches:  "
        f"{basic:,} ({basic / total * 100:.2f}%)"
    )

    print(
        f"Suffix-normalized matches: "
        f"{suffix:,} ({suffix / total * 100:.2f}%)"
    )


# ------------------------------------------------------------
# 11. SHOW REMAINING DIFFERENCES
# ------------------------------------------------------------

remaining = pairs[
    pairs["s1_no_suffix"] != pairs["matched_no_suffix"]
]

print("\n")
print("=" * 60)
print("REMAINING NAME DIFFERENCES")
print("=" * 60)

if len(remaining) == 0:

    print("No remaining differences.")

else:

    sample = remaining.sample(
        min(30, len(remaining)),
        random_state=42
    )

    print(
        sample[
            [
                "source",
                "s1_name",
                "matched_name"
            ]
        ].to_string(index=False)
    )

Loading ground truth...
Total true match pairs: 7,638,365

Loading S1 business names...
S1 names loaded: 2,083,574

Loading S2 business names...
S2 names loaded: 0

Loading S3 business names...
S3 names loaded: 0

Name availability:
S1 names: 7638365
Matched names: 0

Pairs with both names available: 0


NAME NOISE ANALYSIS

Pairs analyzed: 0


REMAINING NAME DIFFERENCES
No remaining differences.
